In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("shasun/tool-wear-detection-in-cnc-mill")

print("Path to dataset files:", path)

100%|██████████| 2.57M/2.57M [00:00<00:00, 82.7MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/shasun/tool-wear-detection-in-cnc-mill/versions/4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

# Define our base starting target
target_dir = '/content/drive/MyDrive/'

print("Scanning Google Drive directory structure...")
found_project_folder = False

for root, dirs, files in os.walk(target_dir):
    # Search for our target project folder or one of the experiment directories
    if 'project_db' in root or 'experiment_01' in dirs or 'experiment_1' in dirs:
        print(f"\n🎯 FOUND MATCHING PATH: {root}")
        print(f"Subdirectories inside this path: {dirs[:5]}")
        print(f"Files immediately inside this path: {files[:5]}")
        found_project_folder = True
        break

if not found_project_folder:
    print("\n❌ Could not find 'project_db' or 'experiment' folders anywhere under MyDrive.")
    print("Let's list the top-level directories in your Drive to find out where it is:")
    try:
        print(os.listdir(target_dir))
    except Exception as e:
        print(f"Error reading root: {e}")

Scanning Google Drive directory structure...

🎯 FOUND MATCHING PATH: /content/drive/MyDrive/project_db
Subdirectories inside this path: []
Files immediately inside this path: ['experiment_03.csv', 'experiment_04.csv', 'experiment_14.csv', 'experiment_15.csv', 'experiment_18.csv']


In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

# 1. SETUP FLAT DIRECTORY PATH
BASE_DIR = '/content/drive/MyDrive/project_db'
TRAIN_CSV_PATH = os.path.join(BASE_DIR, 'train.csv')

# Verify train.csv exists
if not os.path.exists(TRAIN_CSV_PATH):
    raise FileNotFoundError(f"Could not find train.csv at {TRAIN_CSV_PATH}. Please make sure train.csv is uploaded directly inside project_db.")

# 2. LOAD METADATA CONFIGURATIONS
train_df = pd.read_csv(TRAIN_CSV_PATH)
print(f"Loaded train.csv metadata with {len(train_df)} experiment configurations.")

# 3. LOOP AND COMBINE THE FLAT EXPERIMENT CSV FILES
all_readings = []

for idx, row in train_df.iterrows():
    exp_num = int(row['No'])

    # Matching your flat directory file structure exactly: project_db/experiment_01.csv
    file_name = f"experiment_{exp_num:02d}.csv"
    file_path = os.path.join(BASE_DIR, file_name)

    if os.path.exists(file_path):
        df = pd.read_csv(file_path)

        # Relationally link operational details from train.csv to every row
        df['experiment_id'] = exp_num
        df['material'] = row['material']
        df['feedrate_setting'] = row['feedrate']
        df['clamp_pressure'] = row['clamp_pressure']
        df['tool_condition_baseline'] = row['tool_condition']

        # Build systematic timestamp tracking based on 10Hz sequence data
        df['recorded_at'] = pd.date_range(start=f"2026-05-01 {exp_num:02d}:00:00", periods=len(df), freq='100ms')

        all_readings.append(df)
        print(f"✅ Successfully loaded {file_name} ({len(df)} rows mapped).")
    else:
        print(f"⚠️ Warning: File {file_name} was missing from the folder path.")

# Check to ensure your uploaded data was caught by the parser
if len(all_readings) == 0:
    raise ValueError("CRITICAL ERROR: No experiment files could be loaded. Double check filenames inside project_db.")

# Merge operational datasets
master_df = pd.concat(all_readings, ignore_index=True)
print(f"\nData Aggregation Complete! Combined total data points: {len(master_df)} rows.")

# 4. ALIGN LOGICAL COLUMN NAMES AND FILTER FEATURES
features = [
    'X1_ActualVelocity',
    'Y1_ActualVelocity',
    'S1_ActualVelocity',
    'S1_OutputPower',
    'S1_DCBusVoltage'
]

# Quick validation check for column name variances
for f in features:
    if f not in master_df.columns:
        matched = [col for col in master_df.columns if f.split('_')[1] in col]
        if matched:
            master_df.rename(columns={matched[0]: f}, inplace=True)

X = master_df[features].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 5. RUN ML ANOMALY MODELING (Isolation Forest 5% Rate)
print("\nTraining Isolation Forest engine across data repository...")
model = IsolationForest(contamination=0.05, random_state=42)
master_df['anomaly_label'] = model.fit_predict(X_scaled)
master_df['anomaly_score'] = model.decision_function(X_scaled)

anomalies_count = (master_df['anomaly_label'] == -1).sum()
print(f"AI Matrix Diagnostics Concluded: Isolated {anomalies_count} anomalous readings.")

# 6. ISOLATE ALERTS AND ORGANIZE ARCHITECTURAL COLUMNS
anomalies_df = master_df[master_df['anomaly_label'] == -1].copy()

conditions = [
    (anomalies_df['anomaly_score'] < -0.12),
    (anomalies_df['anomaly_score'] < -0.05)
]
choices = ['high', 'medium']
anomalies_df['severity'] = np.select(conditions, choices, default='low')

# Safe fallback lookup for Machining Process stage names
process_col = 'Machining_Process' if 'Machining_Process' in anomalies_df.columns else anomalies_df.columns[-3]

anomalies_df['message'] = anomalies_df.apply(
    lambda r: f"Critical anomaly window observed during process phase: '{r[process_col]}'. Baseline configuration flagged tool as {r['tool_condition_baseline'].upper()}.",
    axis=1
)
anomalies_df['resolved'] = 0

# 7. EXPORT COMPLIANT CSV EXPORTS FOR 3NF TABLES
print("\nExporting synchronized database data source models...")

sensor_export = master_df[[
    'experiment_id', 'recorded_at', 'X1_ActualVelocity', 'S1_DCBusVoltage',
    'S1_ActualVelocity', 'tool_condition_baseline', 'anomaly_score'
]].copy()
sensor_export.columns = ['equipment_id', 'recorded_at', 'vibration', 'temperature', 'spindle_speed', 'tool_wear', 'anomaly_score']

alert_export = anomalies_df[[
    'experiment_id', 'recorded_at', 'severity', 'message', 'resolved'
]].copy()
alert_export.columns = ['equipment_id', 'recorded_at', 'severity', 'message', 'resolved']

# Write back straight to project_db flat directory level without adding indexing artifact bugs
sensor_export.to_csv(os.path.join(BASE_DIR, 'clean_sensor_readings.csv'), index=False)
alert_export.to_csv(os.path.join(BASE_DIR, 'clean_alerts.csv'), index=False)

print("\n🎉 STEP 1 COMPLETE! Check your Google Drive folder for 'clean_sensor_readings.csv' and 'clean_alerts.csv'.")

Loaded train.csv metadata with 18 experiment configurations.
✅ Successfully loaded experiment_01.csv (1055 rows mapped).
✅ Successfully loaded experiment_02.csv (1668 rows mapped).
✅ Successfully loaded experiment_03.csv (1521 rows mapped).
✅ Successfully loaded experiment_04.csv (532 rows mapped).
✅ Successfully loaded experiment_05.csv (462 rows mapped).
✅ Successfully loaded experiment_06.csv (1296 rows mapped).
✅ Successfully loaded experiment_07.csv (565 rows mapped).
✅ Successfully loaded experiment_08.csv (605 rows mapped).
✅ Successfully loaded experiment_09.csv (740 rows mapped).
✅ Successfully loaded experiment_10.csv (1301 rows mapped).
✅ Successfully loaded experiment_11.csv (2314 rows mapped).
✅ Successfully loaded experiment_12.csv (2276 rows mapped).
✅ Successfully loaded experiment_13.csv (2233 rows mapped).
✅ Successfully loaded experiment_14.csv (2332 rows mapped).
✅ Successfully loaded experiment_15.csv (1381 rows mapped).
✅ Successfully loaded experiment_16.csv (602

In [ ]:
!pip install mysql-connector-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 56.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import mysql.connector
import os

# 1. RETRIEVE CLEAN SOURCE DATA MODLES FROM DRIVE
BASE_DIR = '/content/drive/MyDrive/project_db'
SENSOR_CSV = os.path.join(BASE_DIR, 'clean_sensor_readings.csv')
ALERT_CSV = os.path.join(BASE_DIR, 'clean_alerts.csv')

sensors_df = pd.read_csv(SENSOR_CSV)
alerts_df = pd.read_csv(ALERT_CSV)

# 2. DEFINE DATABASE CONNECTION MATRIX
db_config = {
    'host': '210.117.165.104',
    'port': 3306,
    'user': 'dbuser09',                # Updated to match your workbench active session user
    'password': 'dbuser2026',
    'database': 'dbuser09_schema'       # Updated to match your workbench active schema logs
}

print("Connecting to the JBNU MySQL database server...")
conn = mysql.connector.connect(**db_config)
cursor = conn.cursor()
print("Connected successfully! Starting bulk data injection pipeline...\n")

# 3. BULK INGEST INTO SENSOR_READING TABLE
print("Formatting data arrays for 'sensor_reading'...")
sensor_query = """
    INSERT INTO sensor_reading (equipment_id, recorded_at, vibration, temperature, spindle_speed, tool_wear, anomaly_score)
    VALUES (%s, %s, %s, %s, %s, %s, %s)
"""
# Map rows to tuple lists for safe bulk parsing array processing
sensor_records = [tuple(x) for x in sensors_df[['equipment_id', 'recorded_at', 'vibration', 'temperature', 'spindle_speed', 'tool_wear', 'anomaly_score']].to_numpy()]

print(f"Injecting {len(sensor_records)} records into 'sensor_reading' table...")
cursor.executemany(sensor_query, sensor_records)
conn.commit() # Save transaction adjustments
print(f"Success! Saved {cursor.rowcount} rows into sensor_reading table.\n")

# 4. BULK INGEST INTO ALERT TABLE
print("Formatting data arrays for 'alert'...")
alert_query = """
    INSERT INTO alert (equipment_id, triggered_at, severity, message, resolved)
    VALUES (%s, %s, %s, %s, %s)
"""
alert_records = [tuple(x) for x in alerts_df[['equipment_id', 'recorded_at', 'severity', 'message', 'resolved']].to_numpy()]

print(f"Injecting {len(alert_records)} records into 'alert' table...")
cursor.executemany(alert_query, alert_records)
conn.commit() # Save transaction adjustments
print(f"Success! Saved {cursor.rowcount} rows into alert table.\n")

# 5. DISCONNECT SAFELY
cursor.close()
conn.close()
print("STEP 3 COMPLETE! The server database tables are completely populated with your AI dataset logs.")

Connecting to the JBNU MySQL database server...
Connected successfully! Starting bulk data injection pipeline...

Formatting data arrays for 'sensor_reading'...
Injecting 25286 records into 'sensor_reading' table...
Success! Saved 25286 rows into sensor_reading table.

Formatting data arrays for 'alert'...
Injecting 1265 records into 'alert' table...
Success! Saved 1265 rows into alert table.

STEP 3 COMPLETE! The server database tables are completely populated with your AI dataset logs.
